# Raw IFS Frame — Stellar Spectrum

Simulate a raw Liger IFS detector frame for a stellar source.

## Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import h, c

import astropy.units as u
from expecto import get_spectrum

from liger_iris_sim.utils import LIGER_PROPS
from liger_iris_sim.sources import convolve_spectrum
from liger_iris_sim.raw_ifs import simulate_raw_ifs_frame
from liger_iris_drp_resources import load_filters_summary

## Filter and wavelength grid

In [ ]:
filter_name = "KN2"
resolution = 4000
filter_info = load_filters_summary(filter_name)

# Oversampled grid (~4x) for input spectrum
wave_grid_cube = np.linspace(filter_info["wavemin"], filter_info["wavemax"], 2000)

## Prepare stellar spectrum

1. Load a PHOENIX model spectrum from `expecto`
2. Crop it to the filter bandpass
3. Convert it to a photon flux collected by the telescope
4. Convolve it by the instrument's spectral resolution.

In [ ]:
def average_identical_wavelengths(wave, flux):
    unique_wave, indices = np.unique(wave, return_inverse=True)
    avg_flux = np.zeros_like(unique_wave)
    for i in range(len(unique_wave)):
        avg_flux[i] = np.mean(flux[indices == i])
    return unique_wave, avg_flux

def crop_phoenix_spectrum(wave, flux, wavemin, wavemax, pad=0.01):
    pad = pad * (wavemax - wavemin)
    wavemin -= pad
    wavemax += pad
    mask = (wave >= wavemin) & (wave <= wavemax)
    return wave[mask].astype(np.float64), flux[mask].astype(np.float32)

# Retrieve stellar spectrum from PHOENIX library
spectrum = get_spectrum(T_eff=3800, log_g=4.5, Z=0, alpha=0, cache=True)

# Convert wavelength to microns and flux to photons/s/micron
wave_phoenix_um = spectrum.spectral_axis.to(u.micron).value  # microns
flux_phoenix = spectrum.flux.to(u.Joule / u.s / u.m**2 / u.micron).value # J/s/m²/μm
Ephot = h * c / (wave_phoenix_um * 1e-6) # J/photon
flux_phoenix /= Ephot # photons/s/m²/μm
flux_phoenix *= LIGER_PROPS['collarea'] # photons/s/micron

# Crop the PHOENIX spectrum to the wavelength range of the filter
wave_phoenix_um, flux_phoenix = crop_phoenix_spectrum(
    wave_phoenix_um, flux_phoenix, filter_info["wavemin"], filter_info["wavemax"]
)

# Pheonix spectra have some duplicate wavelengths (cause unknown)
# so we need to average the flux values for those wavelengths
wave_phoenix_um, flux_phoenix = average_identical_wavelengths(
    wave_phoenix_um, flux_phoenix
)

# Convolve the PHOENIX spectrum to the instrument resolution
flux_phoenix_convolved = convolve_spectrum(wave_phoenix_um, flux_phoenix, resolution)

## Build the IFS input cube

Generate an IFS cube with identical spectrum and flux level for each lenslet.

Units are phot/s/micron.

In [ ]:
cube_density = np.zeros((len(wave_phoenix_um), 128, 128), dtype=np.float32)
cube_density[:, :, :] = flux_phoenix_convolved[:, None, None]

## Simulate the raw detector frame

`simulate_raw_ifs_frame` renders the lenslet spectra through the trace geometry onto the detector. With `itime=0`, no noise is added and only the noiseless `sim` frame is populated.

In [ ]:

# Read noise = 0 and itime = 0 to simulate a noiseless raw frame.
read_noise = 0
itime = 0

result = simulate_raw_ifs_frame(
    input_cube=cube_density,
    input_wave=wave_phoenix_um,
    ifs_mode='lenslet',
    filter_name=filter_name,
    resolution=resolution,
    itime=itime,
    read_noise=read_noise,
    tracepos_deg=1,
    wavesol_deg=1,
)

## Save to FITS

`save_raw_frame_to_fits` only writes the noisy `DATA` extension, which is `None` here since `itime=0`. Save the noiseless `SIM` frame directly (along with the high-resolution input spectrum, for later comparison against an extracted spectrum).

In [ ]:
from astropy.io import fits

output_dir = "output/"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "simulated_liger_raw_frame_star.fits")

fits.HDUList([
    fits.PrimaryHDU(),
    fits.ImageHDU(result['sim'], name="SIM"),
    fits.ImageHDU(result['data'], name="DATA"),
    fits.ImageHDU(result['error'], name="ERR"),
    fits.ImageHDU(wave_grid_cube, name="WAVE"),
    fits.ImageHDU(wave_phoenix_um, name="WAVEHR"),
    fits.ImageHDU(flux_phoenix_convolved, name="FLUXHR"),
]).writeto(output_path, overwrite=True)

## Visualize the raw frame

Show the full simulated detector frame, with an inset zoom on a 20 × 20 pixel region at the frame center to see the individual lenslet spectral traces up close.

In [ ]:
im = plt.imshow(result['sim'], cmap='viridis')
plt.colorbar(im, label='Signal (phot/s)')
plt.title('Simulated Raw IFS Frame — Stellar Spectrum')
plt.xlabel('Detector column (pixels)')
plt.ylabel('Detector row (pixels)')

plt.tight_layout()
plt.show()